# MEDREx — Phase 2: TF-IDF Baseline
**CSC 590 Masters Project | CSUDH Spring 2026**  
**Student:** Kingslee Dominic Savio Velu | **Mentor:** Dr. Bin Tang

---
### What this notebook does
Trains and evaluates TF-IDF + Logistic Regression for 35-class cancer type classification.
This is Method #1 in the MEDREx 5-method comparison.

**Reference baseline (Cedars-Sinai):** BoW + LR = 95.31% on 32 classes  
**Our goal:** TF-IDF + LR on all 35 classes with class imbalance handling


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')

BASE_DIR     = r'c:\Users\kings\OneDrive\Desktop\CSUDH_Kingslee\CSUDH-Spring-2026\CSC-590-MastersProject\MastersProject'
REPORTS_PATH = os.path.join(BASE_DIR, 'DataSet', 'TCGA_Reports.csv', 'TCGA_Reports.csv')
LABELS_PATH  = os.path.join(BASE_DIR, 'DataSet', 'tcga_patient_to_cancer_type.csv')
print('Libraries loaded.')

## 1. Load & Prepare Data

In [ ]:
df_reports = pd.read_csv(REPORTS_PATH)
df_labels  = pd.read_csv(LABELS_PATH)
df_reports['patient_id'] = df_reports['patient_filename'].apply(lambda x: x.split('.')[0])
df = df_reports.merge(df_labels, on='patient_id', how='inner')
print(f'Dataset: {len(df):,} reports, {df["cancer_type"].nunique()} cancer types')
df[['patient_id', 'cancer_type', 'text']].head(3)

## 2. Train / Validation / Test Split (stratified)

In [ ]:
X = df['text']
y = df['cancer_type']

# 60% train, 20% val, 20% test — stratified
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.40, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f'Train: {len(X_train):,}  Val: {len(X_val):,}  Test: {len(X_test):,}')

## 3. TF-IDF Vectorizer

In [ ]:
# TF-IDF: weighted word importance (improvement over reference's raw word counts)
# sublinear_tf=True: log scaling so one word appearing 100x doesn't dominate
# ngram_range=(1,2): include single words AND two-word phrases (e.g. 'invasive ductal')
vectorizer = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=2,
    strip_accents='unicode',
    analyzer='word'
)

X_train_vec = vectorizer.fit_transform(X_train)
X_val_vec   = vectorizer.transform(X_val)
X_test_vec  = vectorizer.transform(X_test)

print(f'Vocabulary size: {len(vectorizer.vocabulary_):,}')
print(f'Feature matrix (train): {X_train_vec.shape}')

## 4. Train Logistic Regression (Method 1)

In [ ]:
# class_weight='balanced' handles the imbalance problem (BRCA 1097 vs small classes ~50)
print('Training Logistic Regression...')
lr_model = LogisticRegression(
    C=1.0, max_iter=1000,
    class_weight='balanced',
    solver='lbfgs',
    multi_class='multinomial',
    random_state=42
)
lr_model.fit(X_train_vec, y_train)

lr_val_acc  = accuracy_score(y_val,  lr_model.predict(X_val_vec))
lr_test_acc = accuracy_score(y_test, lr_model.predict(X_test_vec))
lr_f1       = f1_score(y_test, lr_model.predict(X_test_vec), average='weighted')

print(f'  Validation accuracy: {lr_val_acc*100:.2f}%')
print(f'  Test accuracy:       {lr_test_acc*100:.2f}%')
print(f'  Weighted F1 score:   {lr_f1*100:.2f}%')

## 5. Train Linear SVM (Method 2)

In [ ]:
print('Training Linear SVM...')
svm_model = LinearSVC(
    C=1.0, max_iter=2000,
    class_weight='balanced',
    random_state=42
)
svm_model.fit(X_train_vec, y_train)

svm_val_acc  = accuracy_score(y_val,  svm_model.predict(X_val_vec))
svm_test_acc = accuracy_score(y_test, svm_model.predict(X_test_vec))
svm_f1       = f1_score(y_test, svm_model.predict(X_test_vec), average='weighted')

print(f'  Validation accuracy: {svm_val_acc*100:.2f}%')
print(f'  Test accuracy:       {svm_test_acc*100:.2f}%')
print(f'  Weighted F1 score:   {svm_f1*100:.2f}%')

## 6. Comparison Table

In [ ]:
results = pd.DataFrame([
    {'Method': 'Reference (BoW + LR, 32 classes)', 'Test Accuracy': 95.31, 'Weighted F1': '~95%', 'Classes': 32, 'Status': 'Reference'},
    {'Method': 'TF-IDF + LR (ours)',  'Test Accuracy': round(lr_test_acc*100,2),  'Weighted F1': f'{lr_f1*100:.2f}%',  'Classes': 35, 'Status': 'Done'},
    {'Method': 'TF-IDF + SVM (ours)', 'Test Accuracy': round(svm_test_acc*100,2), 'Weighted F1': f'{svm_f1*100:.2f}%', 'Classes': 35, 'Status': 'Done'},
    {'Method': 'BERT (Bio_ClinicalBERT)', 'Test Accuracy': None, 'Weighted F1': 'TBD', 'Classes': 35, 'Status': 'Planned'},
    {'Method': 'Embedding Similarity',   'Test Accuracy': None, 'Weighted F1': 'TBD', 'Classes': 35, 'Status': 'Planned'},
    {'Method': 'RAG + LLM (MEDREx)',     'Test Accuracy': None, 'Weighted F1': 'TBD', 'Classes': 35, 'Status': 'Planned'},
])
print(results.to_string(index=False))

In [ ]:
# Accuracy comparison chart
done_methods  = ['Reference\n(BoW+LR 32cls)', 'TF-IDF + LR\n(35 cls)', 'TF-IDF + SVM\n(35 cls)']
done_accuracy = [95.31, lr_test_acc*100, svm_test_acc*100]
colors        = ['#94a3b8', '#1a3a5c', '#2563eb']

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(done_methods, done_accuracy, color=colors, width=0.5, edgecolor='white')
ax.set_ylim(80, 100)
ax.set_ylabel('Test Accuracy (%)')
ax.set_title('MEDREx — TF-IDF Methods vs Reference', fontweight='bold')
for bar, val in zip(bars, done_accuracy):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            f'{val:.1f}%', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'notebooks', 'tfidf_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

## 7. Detailed Classification Report (LR)

In [ ]:
y_pred_lr = lr_model.predict(X_test_vec)
print(classification_report(y_test, y_pred_lr, zero_division=0))

## 8. Confusion Matrix (Top 15 classes)

In [ ]:
top15 = df['cancer_type'].value_counts().head(15).index.tolist()
mask  = y_test.isin(top15)
cm    = confusion_matrix(y_test[mask], y_pred_lr[mask], labels=top15)

fig, ax = plt.subplots(figsize=(14, 11))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=top15, yticklabels=top15, ax=ax,
            linewidths=0.5, linecolor='white')
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('Actual',    fontsize=12)
ax.set_title('Confusion Matrix — TF-IDF + LR (Top 15 Cancer Types)', fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'notebooks', 'confusion_matrix_lr.png'), dpi=150, bbox_inches='tight')
plt.show()

## 9. Per-Class F1 — Which Cancers Are Hard to Predict?

In [ ]:
from sklearn.metrics import f1_score
classes   = sorted(y_test.unique())
f1_scores = f1_score(y_test, y_pred_lr, labels=classes, average=None, zero_division=0)

df_f1 = pd.DataFrame({'cancer_type': classes, 'f1': f1_scores})
df_f1 = df_f1.sort_values('f1')

fig, ax = plt.subplots(figsize=(10, 9))
colors  = ['#ef4444' if f < 0.7 else '#f97316' if f < 0.85 else '#22c55e' for f in df_f1['f1']]
ax.barh(df_f1['cancer_type'], df_f1['f1'], color=colors)
ax.axvline(0.85, color='gray', linestyle='--', alpha=0.7, label='0.85 threshold')
ax.set_xlabel('F1 Score')
ax.set_title('Per-Class F1 Score — TF-IDF + LR', fontweight='bold')
ax.set_xlim(0, 1.05)
ax.legend()
for i, (_, row) in enumerate(df_f1.iterrows()):
    ax.text(row['f1'] + 0.01, i, f'{row["f1"]:.2f}', va='center', fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'notebooks', 'per_class_f1.png'), dpi=150, bbox_inches='tight')
plt.show()

print('Hardest classes to predict:')
print(df_f1.head(5).to_string(index=False))

## 10. Save Model for Demo App

In [ ]:
import joblib

MODEL_DIR = os.path.join(BASE_DIR, 'demo', 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

joblib.dump(lr_model,   os.path.join(MODEL_DIR, 'tfidf_lr_model.pkl'))
joblib.dump(vectorizer, os.path.join(MODEL_DIR, 'tfidf_vectorizer.pkl'))

print(f'Model saved to: {MODEL_DIR}')
print(f'Final test accuracy: {lr_test_acc*100:.2f}%')